In [ ]:
import base64
import csv
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

JOB = "integrity"

os.environ.setdefault("FABRID_JOB_START", str(time.time()))
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    for key in ("GITHUB_TOKEN", "GIT_USER_NAME", "GIT_USER_EMAIL", "REPO_URL"):
        if not os.environ.get(key):
            try:
                os.environ[key] = secrets.get_secret(key)
            except Exception:
                print(f"secret {key} not set")
except ImportError:
    print("kaggle_secrets not available, using environment only")

REPO_URL = os.environ.get("REPO_URL") or "https://github.com/FaiazPronoy/fabrid-benchmark.git"
os.environ["REPO_URL"] = REPO_URL
REPO = Path("/kaggle/working/fabrid-benchmark")

auth = []
if os.environ.get("GITHUB_TOKEN"):
    encoded = base64.b64encode(f"x-access-token:{os.environ['GITHUB_TOKEN']}".encode()).decode()
    auth = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {encoded}"]

if REPO.exists():
    shutil.rmtree(REPO)
cloned = subprocess.run(["git", *auth, "clone", "--quiet", REPO_URL, str(REPO)], check=False)
if cloned.returncode != 0:
    raise RuntimeError("git clone failed")
commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
print("cloned", REPO_URL, "at", commit)

installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[kaggle]"])
if installed.returncode != 0:
    raise RuntimeError("package install failed")

with (REPO / "kaggle" / "jobs.csv").open(newline="") as handle:
    jobs = {row["job"]: row for row in csv.DictReader(handle)}
if JOB not in jobs:
    raise KeyError(f"unknown job {JOB}, choose from {sorted(jobs)}")
job = jobs[JOB]
if job["device"] == "gpu" and shutil.which("nvidia-smi") is None:
    print("warning: job expects a GPU but none is visible")
print("running", JOB, ":", job["command"])
finished = subprocess.Popen(
    job["command"],
    shell=True,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in finished.stdout:
    print(line, end="", flush=True)
finished.wait()
if finished.returncode != 0:
    raise RuntimeError(f"job {JOB} exited with code {finished.returncode}")
print("job", JOB, "finished")